# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rohmasaeed/flyrank-ml-internship-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized (1).csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### 1. Unit of analysis + time window

**One row = one content item for one client.**

The dataset is a content-level SEO and engagement snapshot. Each row contains SEO performance, content characteristics, traffic metrics, ranking metrics, and recent trend information for one `content_id` belonging to one `client_id`.

The dataset contains pre-computed 90-day and last-30-day/previous-30-day performance windows. Therefore, I treat the available observation as a snapshot rather than claiming that the dataset contains one row for every calendar date.

My lane is **Refresh / Content Opportunity Scoring**. The goal is to rank content items that may deserve a refresh based on declining performance and other SEO signals.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### 2. Fields: feature / label / context / excluded

**Features**

I will use these five features:

* `search_volume` — indicates the amount of search demand associated with the content.
* `ctr` — indicates how effectively impressions result in clicks.
* `avg_position` — indicates the page's average search ranking.
* `engagement_rate` — indicates how users engage after arriving on the page.
* `days_since_last_update` — indicates how recently the content was updated.

**Label / proxy**

I will use `trend_pct` as the refresh-opportunity outcome proxy. A strongly negative `trend_pct` indicates declining performance and can be used to identify pages that may deserve a refresh.

**Context**

* `content_id` — identifies the content item.
* `client_id` — identifies the client associated with the content.
* `content_type` — describes the type of content.
* `main_intent` — describes the primary search intent.
* `trend_direction` — provides the categorical direction of recent performance.

**Excluded**

I will exclude label-derived fields such as `trend_pct` from the honest feature set because the label/proxy is derived from the same outcome information. I will also exclude future outcome information because it would not be available at the refresh decision moment.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Verification 1 — Grain

I claim that one row represents one content item for one client. I verify this by comparing the total number of rows with the number of distinct `content_id` and `client_id` combinations.


In [19]:
import pandas as pd
df=pd.read_csv("/content/content_refresh_anonymized (1).csv")
grain_check = pd.DataFrame({
    "total_rows": [len(df)],
    "unique_content_client_pairs": [
        df[["content_id", "client_id"]].drop_duplicates().shape[0]
    ]
})

grain_check

,total_rows,unique_content_client_pairs
0,30000,30000


The total row count matches the number of unique content-client pairs. This supports the claimed grain that one row represents one content item for one client.


### Verification 2 — Row count and performance windows

The dataset contains one row per content-client pair rather than daily observations. I therefore verify the row count and the available performance windows from the provided 90-day, last-30-day, and previous-30-day fields.


In [20]:
window_check = pd.DataFrame({
    "row_count": [len(df)],
    "impressions_90d_available": [
        df["impressions_90d"].notna().sum()
    ],
    "last_30d_impressions_available": [
        df["impressions_last_30d"].notna().sum()
    ],
    "previous_30d_impressions_available": [
        df["impressions_prev_30d"].notna().sum()
    ]
})

window_check

,row_count,impressions_90d_available,last_30d_impressions_available,previous_30d_impressions_available
0,30000,30000,30000,30000


The dataset contains **30,000 content records**. All 30,000 records have non-missing values for `impressions_90d`, `impressions_last_30d`, and `impressions_prev_30d`.

Therefore, all three pre-computed performance windows are available for the observed rows in this dataset. The dataset is a content-level snapshot rather than a daily time-series table, so I do not claim a specific calendar date span.


### Verification 3 — Availability

The reduced dataset does not contain a native Boolean publication or availability field. I therefore define an availability proxy based on the presence of both `content_id` and `client_id`.

The availability check uses a Boolean `True` condition to count rows that satisfy this proxy. This should not be interpreted as an actual publication-status field.


In [21]:
df["is_available"] = (
    df["content_id"].notna()
    & df["client_id"].notna()
)

In [22]:
availability_check = pd.DataFrame({
    "total_rows": [len(df)],
    "available_rows": [
        df["is_available"].eq(True).sum()
    ]
})

availability_check

,total_rows,available_rows
0,30000,30000


### Five-feature frame

I selected five features for refresh opportunity scoring:

| Feature                  | Available when?                                                                                       |
| ------------------------ | ----------------------------------------------------------------------------------------------------- |
| `search_volume`          | Available before the refresh decision because historical search-demand information is already known.  |
| `ctr`                    | Available before the refresh decision because historical click-through performance can be measured.   |
| `avg_position`           | Available before the refresh decision because historical search ranking performance is observed.      |
| `engagement_rate`        | Available before the refresh decision because historical user engagement is already measured.         |
| `days_since_last_update` | Available before the refresh decision because the date of the latest content update is already known. |

These features describe the content's historical demand, search performance, engagement, and freshness without using the refresh outcome itself.


In [23]:
feature_cols = [
    "search_volume",
    "ctr",
    "avg_position",
    "engagement_rate",
    "days_since_last_update"
]

feature_frame = df[
    ["content_id", "client_id"] + feature_cols
].copy()

print("Feature frame shape:", feature_frame.shape)

feature_frame.head()

Feature frame shape: (30000, 7)


,content_id,client_id,search_volume,ctr,avg_position,engagement_rate,days_since_last_update
0,content_304f48230142,client_f369cb89fc,10.0,0.76,10.6,5.88,20
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.05,20.3,0.00,25
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.09,36.5,0.00,20
3,content_331d6c4de07b,client_19581e27de,10.0,0.49,6.2,1.28,22
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.13,44.0,0.00,14


### Refresh opportunity proxy

I define a binary refresh-opportunity proxy using `trend_pct`. A value of -20 or lower indicates that the content experienced a decline of at least 20%.

This is a directional proxy for prioritizing content refresh opportunities, not proof that a refresh will improve future performance.


In [24]:
df["refresh_decline"] = (
    df["trend_pct"] <= -20
).astype(int)

print(df["refresh_decline"].value_counts())

refresh_decline
1    16313
0    13687
Name: count, dtype: int64


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

model_data = df.dropna(
    subset=feature_cols + ["refresh_decline"]
).copy()

X = model_data[feature_cols]
y = model_data["refresh_decline"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_score = accuracy_score(
    y_test,
    honest_pred
)

print("Honest accuracy:", round(honest_score, 3))

Honest accuracy: 0.602


### Honest model result

The honest model achieved an accuracy of **0.602 (60.2%)** using only the five features that are intended to be available before the refresh decision.

This result is treated as a measured, directional result rather than proof that the model will generalize to unseen data. The score provides a baseline for comparing the deliberately leaked version.


### Deliberate leakage experiment

I now intentionally add `trend_pct` to the feature set even though the refresh proxy is directly derived from it. This is deliberate leakage for demonstration purposes.

If the score increases sharply, this shows that the model is receiving information that is too closely connected to the outcome.


In [26]:
leaky_features = feature_cols + ["trend_pct"]

X_leaky = model_data[leaky_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leaky_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

leaky_model.fit(X_train, y_train)

leaky_pred = leaky_model.predict(X_test)

leaky_score = accuracy_score(
    y_test,
    leaky_pred
)

print("Leaky accuracy:", round(leaky_score, 3))

Leaky accuracy: 1.0


In [27]:
comparison = pd.DataFrame({
    "Model": ["Honest", "Leaky"],
    "Accuracy": [honest_score, leaky_score]
})

comparison

,Model,Accuracy
0,Honest,0.602354
1,Leaky,1.000000


### Leakage result and removal

The honest model achieved an accuracy of **0.602 (60.2%)** using only features intended to be available before the refresh decision.

After intentionally adding `trend_pct`, the accuracy increased to **1.000 (100%)**. This large increase demonstrates the leakage trap because the refresh-opportunity proxy was directly derived from `trend_pct`.

The leaky result is not a valid performance estimate because the model was given information derived from the outcome.

I therefore removed `trend_pct` from the final feature set and retained the honest accuracy of **0.602 (60.2%)** as the valid measured result.

**Leakage lesson:** a perfect score can be misleading when a feature contains information from the label or outcome.



In [28]:
final_feature_cols = [
    "search_volume",
    "ctr",
    "avg_position",
    "engagement_rate",
    "days_since_last_update"
]

print("Final features:", final_feature_cols)
print("Removed leakage feature: trend_pct")

Final features: ['search_volume', 'ctr', 'avg_position', 'engagement_rate', 'days_since_last_update']
Removed leakage feature: trend_pct


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### 4. Data limits

**Limitation — pre-aggregated snapshot:** This reduced dataset contains pre-computed 90-day and 30-day performance windows rather than the full daily history. Therefore, it can support directional refresh-priority analysis, but it cannot fully reconstruct daily performance changes or establish that a content refresh caused future performance improvements.

The results should be treated as **decision-support signals**, not causal evidence.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.